# Laya vs Phi-4-mini on ProntoQA: accuracy vs number of reasoning hops

Standalone Colab notebook. It needs no repo checkout: the data files are fetched from GitHub when they aren't next to the notebook.

**Question:** [BoolQ](https://github.com/CJosh88/slms/tree/main/experiments/2026-09-laya-vs-phi4-boolq) asked whether Laya can *find* an answer in a passage. This experiment asks whether it can *chain* facts. Laya decides in one forward pass with no scratchpad, so the hypothesis is that it handles 1-hop questions and falls off as the hop count grows. Phi-4-mini, especially when it reasons step by step, should degrade more slowly. The headline is **accuracy against the number of hops**.

**Why ProntoQA rather than ProofWriter:** the [ProofWriter run](https://github.com/CJosh88/slms/tree/main/experiments/2026-09-laya-vs-phi4-proofwriter) was undermined by a surface cue. There, ~95% of False statements contain "not" and ~95% of True ones don't. ProntoQA was built by its authors to test whether chain-of-thought helps as hops increase. It uses a made-up vocabulary ("Every wumpus is a yumpus"), so no world knowledge helps. Each theory also includes a distractor rule with the opposite property, so merely spotting the property in the text isn't enough.

| Arm | Params | How it answers |
|---|---|---|
| Laya (zero-shot) | 421M | One `noul` (yes/no) question: is the statement true given the facts and rules? One forward pass |
| Laya (fine-tuned) | 421M | The same, after fine-tuning on 3,000 generated *training* examples spread over 1–5 hops |
| Phi-4-mini (direct) | 3.8B | Chat prompt, "Answer with only True or False", greedy decoding |
| Phi-4-mini (CoT) | 3.8B | Chat prompt, "Reason step by step…", then `Answer: True/False`, greedy, up to 1,024 new tokens |

**Data:** generated with the authors' official generator ([`asaparov/prontoqa`](https://github.com/asaparov/prontoqa), Apache-2.0), pinned to commit `0a6412b` by `scripts/generate_data.sh`.
- Settings: fictional ontology, random sentence order, relevant distractors, 1–5 hops.
- The hop count is the generator's own setting (`--min-hops`/`--max-hops`). It is never inferred.
- Test and train use different seeds.
- The test sample has **50 questions for every hop × label × "not"-or-not cell**, 1,000 in total. So chance is exactly 50% at every hop count, and a "not"-based guess scores exactly 50%.

**Runtime → Change runtime type → T4 GPU** before running. Outputs and the fine-tuned checkpoint are cached, so a rerun resumes. Set `USE_DRIVE = True` to keep them on Google Drive. The whole notebook takes roughly 1–1.5 hours on a T4, mostly Phi-4-mini's step-by-step reasoning.

In [ ]:
%pip install -q "laya==0.3.21" "transformers==4.57.3" "accelerate==1.11.0" safetensors scikit-learn scipy statsmodels

In [ ]:
import gc, json, math, os, random, re, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch, transformers, laya
from tqdm.auto import tqdm

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print("laya", laya.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__)
assert torch.cuda.is_available(), "Fine-tuning needs a GPU: Runtime -> Change runtime type -> T4 GPU"
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))
# T4 (compute capability 7.5) has no native bf16, so use fp16 there.
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

## 1. Config

In [ ]:
HOPS = [1, 2, 3, 4, 5]
LABELS = ["True", "False"]
N_PER_CELL = 50               # test questions per (hops, label, has "not") cell -> 5 x 2 x 2 x 50 = 1,000
THRESHOLD = 0.5               # True when P(true) >= THRESHOLD; a tuned threshold is also reported

# Laya fine-tuning (the single-T4 port of the official recipe from the BoolQ fine-tuning experiment)
N_TRAIN_PER_CELL = 300        # per (hops, label) from the train files -> 3,000 items
N_CALIB_PER_CELL = 30         # per (hops, label), disjoint from the above -> 300 items, for temperature + thresholds
LAYA_ID = "convaiinnovations/laya"
EPOCHS = 3
MICRO_BATCH = 8
GRAD_ACCUM = 8                # effective batch = MICRO_BATCH * GRAD_ACCUM = 64
GROUP_SIZE = 4                # noisy samples per item for the policy-gradient baseline
LR_ENCODER = 2.5e-5
LR_HEAD = 1.0e-4
SIGMA_START, SIGMA_END = 0.4, 0.1

# Phi-4-mini
PHI_ID = "microsoft/Phi-4-mini-instruct"
PHI_BATCH = 8                 # lower this if you hit CUDA out-of-memory errors
COT_MAX_NEW_TOKENS = 1024

# One definition of the task, shared by every model.
LAYA_INSTRUCTIONS = "Using only the facts and rules in the text, is this statement true? Statement: {statement}"
LAYA_CRITERIA = {"true": "the facts and rules prove the statement",
                 "false": "the facts and rules prove the statement is false"}
SYSTEM_PROMPT = ("You judge whether a statement is true or false using only the facts and rules given. "
                 "The words are made up, so rely on the given rules, not on outside knowledge.")
DIRECT_TEMPLATE = "Facts and rules:\n{theory}\n\nStatement: {statement}\nAnswer with only True or False."
COT_TEMPLATE = (
    "Facts and rules:\n{theory}\n\nStatement: {statement}\n"
    "Reason step by step, citing the facts and rules you use. Then finish with a final line of the form "
    "\"Answer: True\" or \"Answer: False\"."
)

# Data generated by scripts/generate_data.sh (official generator, pinned commit); fetched when not local.
DATA_URL = "https://raw.githubusercontent.com/CJosh88/slms/main/experiments/2026-09-laya-vs-phi4-prontoqa/data/"

USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/slms/2026-09-laya-vs-phi4-prontoqa")
else:
    BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
CACHE_DIR = BASE_DIR / "cache"
FIG_DIR = BASE_DIR / "figures"
FT_DIR = BASE_DIR / "laya_prontoqa_ft"   # fine-tuned checkpoint (~850 MB, not committed)
for d in (DATA_DIR, CACHE_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 2. Load ProntoQA

Each generated file holds one hop count. Each example has the `question` (the theory: facts, rules and distractors), a `query` ("True or false: …"), the gold `chain_of_thought` and the `answer`. The hop count comes from the file, which is the generator's `--min-hops`/`--max-hops` setting. As a sanity check (not a recomputation), each gold proof must have exactly `2 × hops + 1` steps.

In [ ]:
import urllib.request


def load_file(split, hops):
    path = DATA_DIR / f"{split}_{hops}hop.json"
    if not path.exists():
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    rows = []
    for key, ex in json.loads(path.read_text(encoding="utf-8")).items():
        t = ex["test_example"]
        assert t["query"].startswith("True or false: "), t["query"]
        assert t["answer"] in LABELS, t["answer"]
        assert len(t["chain_of_thought"]) == 2 * hops + 1, (split, hops, key)
        statement = t["query"][len("True or false: "):]
        rows.append({"id": f"{split}-{hops}hop-{key}", "hops": hops, "theory": t["question"], "statement": statement,
                     "label": t["answer"], "neg": " not " in f" {statement} ", "gold_cot": " ".join(t["chain_of_thought"])})
    return rows


def load_split(split):
    return pd.DataFrame([r for h in HOPS for r in load_file(split, h)])


def stratified(d, by, per_cell, name, exclude=()):
    d = d[~d.id.isin(set(exclude))]
    parts = []
    for key, g in d.groupby(by):
        if len(g) < per_cell:
            raise ValueError(f"{name}: cell {dict(zip(by, key))} has only {len(g)} questions (< {per_cell})")
        parts.append(g.sample(n=per_cell, random_state=SEED))
    out = pd.concat(parts).sort_values(["hops", "label", "id"]).reset_index(drop=True)
    return out


full_test, full_train = load_split("test"), load_split("train")
print("Generated test examples (hops x label x has 'not'):")
print(full_test.groupby(["hops", "label", "neg"]).size().unstack([1, 2]))

test_df = stratified(full_test, ["hops", "label", "neg"], N_PER_CELL, "test")
train_df = stratified(full_train, ["hops", "label"], N_TRAIN_PER_CELL, "train")
calib_df = stratified(full_train, ["hops", "label"], N_CALIB_PER_CELL, "calib", exclude=train_df.id)

# Different seeds make overlap unlikely, but check: no test theory+statement may appear in train/calib.
key = lambda d: set(d.theory + " || " + d.statement)
assert not key(test_df) & (key(train_df) | key(calib_df)), "test example also in train/calib"
assert not set(train_df.id) & set(calib_df.id)

print(f"\nTest: {len(test_df)} | train: {len(train_df)} | calibration: {len(calib_df)}")
print(test_df.groupby(["hops", "label", "neg"]).size().unstack([1, 2]))
r = test_df[test_df.hops == 3].iloc[0]
print(f"\n[{r.id}] {r.hops} hops, gold={r.label}\n{r.theory}\nStatement: {r.statement}\nGold proof: {r.gold_cot}")

## 3. Laya setup and input lengths

Laya's English checkpoint has a 512-token window, shared by the question head and the theory. Theories are not truncated. Any input that fills the window is flagged as `overflow` and reported.

In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, render_options, QTYPES

model_dir = snapshot_download(LAYA_ID)
_fix_tokenizer_config(model_dir)
tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    base_cfg = json.load(f)
MAX_LEN, HEAD_MAX_LEN = base_cfg["max_len"], base_cfg["head_max_len"]
print({"max_len": MAX_LEN, "head_max_len": HEAD_MAX_LEN}, "| QTYPES", QTYPES)


def laya_question(statement):
    # Public API form, for agent.predict
    return {"type": "noul", "instructions": LAYA_INSTRUCTIONS.format(statement=statement), "criteria": LAYA_CRITERIA}


def internal_question(statement):
    # Internal form, for build_sequence (same text as laya_question). Option order is [false, true].
    return {"t": "noul", "ins": LAYA_INSTRUCTIONS.format(statement=statement), "crit": LAYA_CRITERIA}


print(render_options(internal_question("x")))
for d in (test_df, train_df, calib_df):
    d["input_tokens"] = [len(build_sequence(tok, t, internal_question(s), MAX_LEN, HEAD_MAX_LEN)[0])
                         for t, s in zip(d.theory, d.statement)]
    d["overflow"] = d.input_tokens >= MAX_LEN
print(test_df.groupby("hops").input_tokens.describe()[["mean", "50%", "max"]].round(0))
print(f"\nInputs that fill the {MAX_LEN}-token window: test {test_df.overflow.sum()}, "
      f"train {train_df.overflow.sum()}, calib {calib_df.overflow.sum()}")

fig, ax = plt.subplots(figsize=(7, 3.5))
for h in HOPS:
    ax.hist(test_df[test_df.hops == h].input_tokens, bins=30, histtype="step", label=f"{h} hops")
ax.axvline(MAX_LEN, color="crimson", ls="--", label=f"Laya window ({MAX_LEN})")
ax.set_xlabel("Laya input tokens (question head + theory)"); ax.set_ylabel("Questions"); ax.legend(fontsize=8)
ax.set_title("ProntoQA input lengths by hop count (test sample)")
fig.savefig(FIG_DIR / "token_lengths.png", dpi=150, bbox_inches="tight"); plt.show()

## 4. Run Laya (zero-shot)

Each question is one `noul` question, and Laya returns P(true). Examples are scored one at a time (batch size 1), which is Laya's realistic single-request latency. The calibration slice is scored too, so a decision threshold can be tuned on data that isn't the test set.

In [ ]:
def load_cache(path):
    if not path.exists():
        return {}
    with path.open(encoding="utf-8") as f:
        return {rec["id"]: rec for rec in map(json.loads, f)}


def sync():
    torch.cuda.synchronize()


def run_laya(agent, d, stem, desc):
    path, meta_path = CACHE_DIR / f"{stem}.jsonl", CACHE_DIR / f"{stem}_meta.json"
    done = load_cache(path)
    todo = d[~d.id.isin(done)]
    print(f"{desc}: {len(done)} cached, {len(todo)} to run")
    if len(todo):
        torch.cuda.reset_peak_memory_stats()
        with path.open("a", encoding="utf-8") as f:
            for r in tqdm(todo.itertuples(), total=len(todo), desc=desc):
                sync(); t0 = time.perf_counter()
                p = agent.predict(r.theory, {"q": laya_question(r.statement)})["answers"]["q"]["noul"]
                sync()
                f.write(json.dumps({"id": r.id, "p_true": float(p), "latency_s": time.perf_counter() - t0}) + "\n")
        meta_path.write_text(json.dumps({"peak_mem_gb": torch.cuda.max_memory_allocated() / 1e9, "params_m": 421.3}))
    return json.loads(meta_path.read_text())


agent = laya.load(LAYA_ID, device=DEVICE)

# Smoke test on hand-written fictional theories: 1 hop and 2 hops, one True and one False each.
smoke = [
    ("Every wumpus is a yumpus. Yumpuses are red. Every zumpus is not red. Max is a wumpus.", "Max is a yumpus.", "True"),
    ("Every wumpus is a yumpus. Yumpuses are red. Every zumpus is not red. Max is a wumpus.", "Max is red.", "True"),
    ("Every wumpus is a yumpus. Yumpuses are not red. Every zumpus is red. Max is a wumpus.", "Max is red.", "False"),
    ("Every wumpus is a yumpus. Yumpuses are not red. Every zumpus is red. Max is a zumpus.", "Max is not red.", "False"),
]
for theory, statement, gold in smoke:
    p = agent.predict(theory, {"q": laya_question(statement)})["answers"]["q"]["noul"]
    print(f"{statement:18s} gold={gold:5s} P(true)={p:.3f}")

laya_meta = run_laya(agent, test_df, "laya", "Laya zero-shot")
run_laya(agent, calib_df, "laya_calib", "Laya zero-shot (calib)")
print(laya_meta)
del agent; gc.collect(); torch.cuda.empty_cache()

## 5. Fine-tune Laya on ProntoQA

This is the single-T4 port of Laya's [official RLCD recipe](https://github.com/NandhaKishorM/laya/blob/main/notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb), identical to the [BoolQ fine-tuning experiment](https://github.com/CJosh88/slms/tree/main/experiments/2026-09-laya-finetune-boolq): a `noul` target of [false, true] and the same hyperparameters.
- Training uses 3,000 generated *training* examples (a different generator seed from the test set), balanced over 1–5 hops × label.
- A disjoint 300-example calibration slice tracks accuracy per epoch, fits the `noul` temperature and tunes the thresholds.

The question this answers: if Laya's accuracy still falls with hop count after it has seen plenty of 4- and 5-hop examples, the single forward pass is the limit, not a lack of exposure to the task.

In [ ]:
def build_item(theory, statement, label):
    seq, markers = build_sequence(tok, theory, internal_question(statement), MAX_LEN, HEAD_MAX_LEN)
    if len(markers) != 2 or len(seq) >= MAX_LEN:   # drop rather than train on a cut theory
        return None
    y = label == "True"
    return {"ids": seq, "markers": markers, "qtype": QTYPES["noul"], "target": [0.0, 1.0] if y else [1.0, 0.0],
            "label": int(y)}


def build_items(d):
    items = [build_item(t, s, y) for t, s, y in zip(d.theory, d.statement, d.label)]
    kept = [it for it in items if it is not None]
    print(f"  built {len(kept)} / {len(items)} items ({len(items) - len(kept)} dropped), "
          f"max length {max(len(it['ids']) for it in kept)} tokens")
    return kept


print("train:"); train_items = build_items(train_df)
print("calib:"); calib_items = build_items(calib_df)

In [ ]:
from safetensors.torch import load_file, save_file
from laya.common import build_model, clamp_temperature, proper_reward

K = 2


def collate(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, :len(it["ids"])] = torch.tensor(it["ids"])
        att[i, :len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, :k] = torch.tensor(it["target"])
    return {"input_ids": ids, "attention_mask": att, "marker_pos": mpos, "marker_mask": mmask,
            "target": target, "qtype": torch.tensor([it["qtype"] for it in items])}


def forward(model, batch):
    with torch.autocast("cuda", dtype=torch.float16):
        logits, act = model(*(batch[k].to(DEVICE) for k in
                              ("input_ids", "attention_mask", "marker_pos", "marker_mask", "qtype")))
    return logits.float(), act


@torch.no_grad()
def raw_logits(model, items, bs=16):
    model.eval()
    out = []
    for i in range(0, len(items), bs):
        logits, _ = forward(model, collate(items[i:i + bs], tok.pad_token_id))
        out.append(logits[:, :K].cpu())
    model.train()
    return torch.cat(out)


def fit_temperature(z, target):
    # Single temperature minimising cross-entropy on held-out items (as in the official recipe).
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(target * torch.log_softmax(z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(log_t.exp())


ft_done = (FT_DIR / "rl_agent_config.json").exists()
history = json.loads((CACHE_DIR / "train_history.json").read_text()) if ft_done else []
if not ft_done:
    cfg = dict(base_cfg)
    cfg["gradient_checkpointing"] = True
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    model.load_state_dict(load_file(os.path.join(model_dir, "model.safetensors")), strict=True)
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(DEVICE).train()

    enc_params = [p for n, p in model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in model.named_parameters() if "encoder." not in n]
    optimizer = torch.optim.AdamW([{"params": enc_params, "lr": LR_ENCODER},
                                   {"params": head_params, "lr": LR_HEAD}], weight_decay=0.01)
    total_updates = math.ceil(len(train_items) / (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_updates, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda")
    calib_y = torch.tensor([it["label"] for it in calib_items])
    torch.cuda.reset_peak_memory_stats()
    t_start = time.time()

    for epoch in range(EPOCHS):
        rng = random.Random(SEED + epoch)
        items = train_items[:]
        rng.shuffle(items)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * epoch / max(1, EPOCHS - 1)
        optimizer.zero_grad(set_to_none=True)
        losses, rewards = [], []
        n_micro = math.ceil(len(items) / MICRO_BATCH)
        for step, b in enumerate(tqdm(range(0, len(items), MICRO_BATCH), desc=f"epoch {epoch + 1}/{EPOCHS}")):
            batch = collate(items[b:b + MICRO_BATCH], tok.pad_token_id)
            logits, act = forward(model, batch)
            mask = batch["marker_mask"].to(DEVICE)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(DEVICE)
            qtype = batch["qtype"].to(DEVICE)

            # Zero-mean Gaussian exploration on the logits, scored with a proper scoring rule
            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=DEVICE) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), qtype, mask, w_sph=0.75, w_rps=1.0)
                adv = (r - r.mean(0, keepdim=True)) / ((r - r.mean(0, keepdim=True)).std() + 1e-6)
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + loss_ce) / GRAD_ACCUM + 0.0 * act.sum()

            scaler.scale(loss).backward()
            if (step + 1) % GRAD_ACCUM == 0 or step + 1 == n_micro:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update(); scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            losses.append(loss_ce.item()); rewards.append(r.mean().item())

        zc = raw_logits(model, calib_items)
        calib_acc = (zc.argmax(-1) == calib_y).float().mean().item()
        history.append({"epoch": epoch + 1, "ce_loss": float(np.mean(losses)), "reward": float(np.mean(rewards)),
                        "calib_acc": calib_acc, "minutes": (time.time() - t_start) / 60})
        print(history[-1])

    train_minutes = (time.time() - t_start) / 60
    train_peak_gb = torch.cuda.max_memory_allocated() / 1e9

    # Temperature calibration on the held-out slice, then save in the layout laya.Agent loads.
    # The agent picks the `noul:2` bucket for a yes/no question, so the fit goes there too.
    zc = raw_logits(model, calib_items)
    t_noul = clamp_temperature(fit_temperature(zc, torch.tensor([it["target"] for it in calib_items])))
    temps = list(base_cfg.get("temperature", [1.0, 1.0, 1.0]))
    temps[QTYPES["noul"]] = t_noul
    by_options = {**base_cfg.get("temperature_by_options", {}), "noul:2": t_noul}
    print(f"Fitted noul temperature: {t_noul:.3f}")

    FT_DIR.mkdir(parents=True, exist_ok=True)
    save_file({k: v.half().contiguous().cpu() for k, v in model.state_dict().items()},
              str(FT_DIR / "model.safetensors"))
    model.encoder.config.save_pretrained(FT_DIR / "encoder")
    tok.save_pretrained(FT_DIR / "tokenizer")
    save_cfg = {**base_cfg, "fine_tuned": True, "model_name": "laya-prontoqa", "temperature": temps,
                "temperature_by_options": by_options}
    (FT_DIR / "rl_agent_config.json").write_text(json.dumps(save_cfg, indent=2))
    (CACHE_DIR / "train_history.json").write_text(json.dumps(history))
    (CACHE_DIR / "train_meta.json").write_text(json.dumps(
        {"train_minutes": train_minutes, "train_peak_gb": train_peak_gb, "noul_temperature": t_noul,
         "n_train_items": len(train_items), "n_calib_items": len(calib_items), "epochs": EPOCHS}))
    del model, optimizer, scheduler, scaler
    gc.collect(); torch.cuda.empty_cache()

train_meta = json.loads((CACHE_DIR / "train_meta.json").read_text())
print(train_meta)
pd.DataFrame(history)

In [ ]:
ft_agent = laya.Agent(str(FT_DIR), device=DEVICE)
laya_ft_meta = run_laya(ft_agent, test_df, "laya_ft", "Laya fine-tuned")
run_laya(ft_agent, calib_df, "laya_ft_calib", "Laya fine-tuned (calib)")
print(laya_ft_meta)
del ft_agent; gc.collect(); torch.cuda.empty_cache()

## 6. Run Phi-4-mini: direct, then step by step

**Direct:** greedy decoding with `max_new_tokens=3`. P(true) is the softmax over the first answer token's *True* vs *False* logits, pooling spelling variants. The decoded text gives the prediction, falling back to P(true) ≥ 0.5 when it doesn't start with a label. The calibration slice is scored too.

**Step by step (CoT):** greedy decoding with up to `COT_MAX_NEW_TOKENS` new tokens. The last `Answer: …` is the prediction. An output with no parseable answer counts as wrong, and the parse rate is reported. Latency is the batch's wall time divided by the batch size.

In [ ]:
from transformers import AutoModelForCausalLM

phi = {}   # loaded lazily, only if something is left to generate


def get_phi():
    if not phi:
        ptok = AutoTokenizer.from_pretrained(PHI_ID)
        ptok.padding_side = "left"
        if ptok.pad_token is None:
            ptok.pad_token = ptok.eos_token
        model = AutoModelForCausalLM.from_pretrained(PHI_ID, dtype=DTYPE, device_map="auto").eval()
        eos = model.generation_config.eos_token_id
        stop_ids = set(eos if isinstance(eos, list) else [eos]) | {ptok.pad_token_id}
        phi.update(tok=ptok, model=model, stop_ids=stop_ids)
        torch.cuda.reset_peak_memory_stats()
    return phi


def prompts_for(ptok, batch, template):
    return [ptok.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": template.format(theory=t, statement=s)}],
        tokenize=False, add_generation_prompt=True) for t, s in zip(batch.theory, batch.statement)]


def first_token_ids(ptok, words):
    return sorted({ptok.encode(w, add_special_tokens=False)[0] for w in words})


def parse_direct(text):
    t = text.strip().lower().lstrip("*'\"").strip()
    return "True" if t.startswith("true") else "False" if t.startswith("false") else None


ANSWER_RE = re.compile(r"answer\s*(?:is)?\s*[:：]?\s*\**\s*(true|false)\b", re.IGNORECASE)


def parse_cot(text):
    found = ANSWER_RE.findall(text)
    return found[-1].capitalize() if found else None


def longest_first(d):
    # Similar lengths in each batch; a CUDA OOM shows up in the first batch rather than the last.
    return d.assign(n=d.theory.str.len()).sort_values("n", ascending=False)


def run_phi_direct(d, stem):
    path = CACHE_DIR / f"{stem}.jsonl"
    todo = longest_first(d[~d.id.isin(load_cache(path))])
    print(f"{stem}: {len(d) - len(todo)} cached, {len(todo)} to run")
    if not len(todo):
        return
    P = get_phi(); ptok, model = P["tok"], P["model"]
    true_ids = first_token_ids(ptok, ["True", " True", "true", " true", "TRUE"])
    false_ids = first_token_ids(ptok, ["False", " False", "false", " false", "FALSE"])
    assert not set(true_ids) & set(false_ids), (true_ids, false_ids)
    with path.open("a", encoding="utf-8") as f:
        for start in tqdm(range(0, len(todo), PHI_BATCH), desc=stem):
            batch = todo.iloc[start:start + PHI_BATCH]
            enc = ptok(prompts_for(ptok, batch, DIRECT_TEMPLATE), return_tensors="pt", padding=True).to(model.device)
            sync(); t0 = time.perf_counter()
            with torch.no_grad():
                out = model.generate(**enc, max_new_tokens=3, do_sample=False, output_scores=True,
                                     return_dict_in_generate=True, pad_token_id=ptok.pad_token_id)
            sync(); per = (time.perf_counter() - t0) / len(batch)
            logits = out.scores[0].float()
            p_true = torch.sigmoid(torch.logsumexp(logits[:, true_ids], -1) - torch.logsumexp(logits[:, false_ids], -1))
            texts = ptok.batch_decode(out.sequences[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            for id_, t, p in zip(batch.id, texts, p_true.tolist()):
                f.write(json.dumps({"id": id_, "output": t, "parsed": parse_direct(t) is not None, "p_true": p,
                                    "latency_s": per}) + "\n")


run_phi_direct(test_df, "phi4_direct")
run_phi_direct(calib_df, "phi4_direct_calib")

In [ ]:
cot_path = CACHE_DIR / "phi4_cot.jsonl"
todo = longest_first(test_df[~test_df.id.isin(load_cache(cot_path))])
print(f"Phi-4-mini CoT: {len(test_df) - len(todo)} cached, {len(todo)} to run")
if len(todo):
    P = get_phi(); ptok, model, stop_ids = P["tok"], P["model"], P["stop_ids"]
    with cot_path.open("a", encoding="utf-8") as f:
        for start in tqdm(range(0, len(todo), PHI_BATCH), desc="Phi-4-mini CoT"):
            batch = todo.iloc[start:start + PHI_BATCH]
            enc = ptok(prompts_for(ptok, batch, COT_TEMPLATE), return_tensors="pt", padding=True).to(model.device)
            sync(); t0 = time.perf_counter()
            with torch.no_grad():
                seqs = model.generate(**enc, max_new_tokens=COT_MAX_NEW_TOKENS, do_sample=False,
                                      pad_token_id=ptok.pad_token_id)
            sync(); per = (time.perf_counter() - t0) / len(batch)
            gen = seqs[:, enc["input_ids"].shape[1]:].tolist()
            for id_, g in zip(batch.id, gen):
                n_out = next((i for i, t in enumerate(g) if t in stop_ids), len(g))
                text = ptok.decode(g[:n_out], skip_special_tokens=True)
                pred = parse_cot(text)
                f.write(json.dumps({"id": id_, "output": text, "parsed": pred is not None, "pred": pred,
                                    "output_tokens": n_out, "hit_limit": n_out >= COT_MAX_NEW_TOKENS,
                                    "latency_s": per}) + "\n")

phi_meta_path = CACHE_DIR / "phi4_meta.json"
if phi:
    phi_meta_path.write_text(json.dumps({"peak_mem_gb": torch.cuda.max_memory_allocated() / 1e9,
                                         "params_m": sum(p.numel() for p in phi["model"].parameters()) / 1e6}))
    phi.clear(); gc.collect(); torch.cuda.empty_cache()
phi_meta = json.loads(phi_meta_path.read_text())
print(phi_meta)

## 7. Results

Accuracy is reported two ways for the three arms that output P(true):
- at the default threshold of 0.5;
- at a threshold tuned on the calibration slice (never on the test set).

The tuned threshold separates *no signal* from *a signal with a yes/no bias*. ROC-AUC per hop count measures the same thing without any threshold. CoT produces only a label, so it gets a single accuracy.

In [ ]:
from sklearn.metrics import roc_auc_score


def wilson(k, n, z=1.96):
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return centre - half, centre + half


def best_threshold(p, y):
    ths = np.round(np.arange(0.02, 0.99, 0.01), 2)
    return float(ths[np.argmax([((p >= t) == y).mean() for t in ths])])


MODELS = {
    "Laya (zero-shot)": ("laya", laya_meta),
    "Laya (fine-tuned)": ("laya_ft", laya_ft_meta),
    "Phi-4-mini (direct)": ("phi4_direct", phi_meta),
    "Phi-4-mini (CoT)": ("phi4_cot", phi_meta),
}
frames, tuned = [], {}
for name, (stem, _) in MODELS.items():
    g = test_df[["id", "hops", "label", "neg", "overflow"]].merge(
        pd.DataFrame(load_cache(CACHE_DIR / f"{stem}.jsonl").values()), on="id")
    g["y"] = g.label == "True"
    if stem == "phi4_cot":
        g["pred"] = g.pred.fillna("none")
        g["pred_tuned"] = g.pred
    else:
        c = calib_df[["id", "label"]].merge(pd.DataFrame(load_cache(CACHE_DIR / f"{stem}_calib.jsonl").values()), on="id")
        tuned[name] = best_threshold(c.p_true.values, (c.label == "True").values)
        if stem == "phi4_direct":
            parsed = g.output.map(parse_direct)
            g["pred"] = [a if a is not None else ("True" if p >= THRESHOLD else "False") for a, p in zip(parsed, g.p_true)]
        else:
            g["pred"] = np.where(g.p_true >= THRESHOLD, "True", "False")
        g["pred_tuned"] = np.where(g.p_true >= tuned[name], "True", "False")
    g["model"] = name
    frames.append(g)
res = pd.concat(frames, ignore_index=True)
res["correct"] = res.pred == res.label
res["correct_tuned"] = res.pred_tuned == res.label
assert res.groupby("model").size().eq(len(test_df)).all(), res.groupby("model").size()
res.to_csv(BASE_DIR / "results.csv", index=False)
print("Thresholds tuned on the calibration slice:", {k: round(v, 2) for k, v in tuned.items()})

# Shortcut check: "False if the statement says 'not', otherwise True". Balanced sampling makes this exactly 50%.
shortcut = (np.where(test_df.neg, "False", "True") == test_df.label)
print(f"'not' shortcut accuracy: {shortcut.mean():.1%} (should be 50%)")

rows = []
for name, g in res.groupby("model", sort=False):
    for h, gh in g.groupby("hops"):
        lo, hi = wilson(gh.correct.sum(), len(gh))
        rows.append({"model": name, "hops": h, "n": len(gh), "accuracy": gh.correct.mean(), "ci_low": lo, "ci_high": hi,
                     "accuracy_tuned": gh.correct_tuned.mean(),
                     "roc_auc": roc_auc_score(gh.y, gh.p_true) if "p_true" in gh and gh.p_true.notna().all() else float("nan"),
                     "acc_true": gh[gh.y].correct.mean(), "acc_false": gh[~gh.y].correct.mean()})
by_hops = pd.DataFrame(rows)
by_hops.to_csv(BASE_DIR / "by_hops.csv", index=False)
for col in ["accuracy", "accuracy_tuned", "roc_auc"]:
    print(f"\n{col} by hop count")
    display(by_hops.pivot(index="hops", columns="model", values=col)[list(MODELS)].style.format("{:.3f}", na_rep="–"))

rows = [{"model": "Chance", "n": len(test_df), "accuracy": 0.5},
        {"model": "'not' shortcut", "n": len(test_df), "accuracy": shortcut.mean()}]
for name, g in res.groupby("model", sort=False):
    lo, hi = wilson(g.correct.sum(), len(g))
    meta = MODELS[name][1]
    rows.append({
        "model": name, "n": len(g), "accuracy": g.correct.mean(), "ci_low": lo, "ci_high": hi,
        "accuracy_tuned": g.correct_tuned.mean(), "threshold_tuned": tuned.get(name, float("nan")),
        "acc_1hop": g[g.hops == 1].correct.mean(), "acc_5hop": g[g.hops == 5].correct.mean(),
        "roc_auc": roc_auc_score(g.y, g.p_true) if g.p_true.notna().all() else float("nan"),
        "pred_true_rate": (g.pred == "True").mean(),
        "parse_rate": g.parsed.mean() if g.parsed.notna().all() else float("nan"),
        "mean_output_tokens": g.output_tokens.mean() if g.output_tokens.notna().any() else float("nan"),
        "ms_per_example": 1000 * g.latency_s.mean(), "params_m": meta["params_m"], "peak_mem_gb": meta["peak_mem_gb"],
    })
summary = pd.DataFrame(rows).set_index("model")
summary.to_csv(BASE_DIR / "summary.csv")
cot = res[res.model == "Phi-4-mini (CoT)"]
print(f"\nInputs over Laya's window: {test_df.overflow.sum()} | CoT cut off at {COT_MAX_NEW_TOKENS} tokens: "
      f"{cot.hit_limit.mean():.1%} | CoT accuracy when parsed: {cot[cot.parsed == True].correct.mean():.1%}")
print(f"Laya fine-tuning: {train_meta['train_minutes']:.1f} min, peak {train_meta['train_peak_gb']:.1f} GB, "
      f"{train_meta['n_train_items']} items x {train_meta['epochs']} epochs")
summary.style.format({
    "accuracy": "{:.1%}", "ci_low": "{:.1%}", "ci_high": "{:.1%}", "accuracy_tuned": "{:.1%}", "threshold_tuned": "{:.2f}",
    "acc_1hop": "{:.1%}", "acc_5hop": "{:.1%}", "roc_auc": "{:.3f}", "pred_true_rate": "{:.1%}", "parse_rate": "{:.1%}",
    "mean_output_tokens": "{:.0f}", "ms_per_example": "{:.1f}", "params_m": "{:,.0f}", "peak_mem_gb": "{:.2f}",
}, na_rep="–")

### Does Laya degrade faster?

- **Per arm:** a logistic regression of correctness on hop count gives each arm's slope, the change in log-odds of a correct answer per extra hop. More negative means faster degradation.
- **Between arms:** one GEE model fitted across all arms, with a `hops × arm` interaction and clustering on question (every arm answers the same questions). It tests whether the slopes differ.
  - It is fitted twice: with zero-shot Laya as the reference arm, then with fine-tuned Laya.
  - A positive interaction means that arm degrades *less steeply* than the reference.

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

rows = []
for name, g in res.groupby("model", sort=False):
    d = g.assign(yc=g.correct.astype(int))
    try:
        fit = smf.logit("yc ~ hops", data=d).fit(disp=0)
        lo, hi = fit.conf_int().loc["hops"]
        rows.append({"model": name, "slope_log_odds_per_hop": fit.params["hops"], "ci_low": lo, "ci_high": hi,
                     "p": fit.pvalues["hops"]})
    except Exception as e:   # e.g. perfect separation when an arm is right on every question
        rows.append({"model": name, "slope_log_odds_per_hop": float("nan"), "note": str(e)[:80]})
slopes = pd.DataFrame(rows).set_index("model")
display(slopes.style.format({"slope_log_odds_per_hop": "{:+.3f}", "ci_low": "{:+.3f}", "ci_high": "{:+.3f}",
                             "p": "{:.2g}"}, na_rep="–"))

inter = {}
for ref in ["Laya (zero-shot)", "Laya (fine-tuned)"]:
    order = [ref] + [m for m in MODELS if m != ref]
    d = res.assign(yc=res.correct.astype(int), arm=pd.Categorical(res.model, categories=order))
    gee = smf.gee("yc ~ hops * arm", groups="id", data=d, family=sm.families.Binomial(),
                  cov_struct=sm.cov_struct.Exchangeable()).fit()
    t = pd.DataFrame({"slope_diff": gee.params, "p": gee.pvalues}).filter(like="hops:", axis=0)
    t.index = [i.split("[T.")[1].rstrip("]") for i in t.index]
    inter[f"vs {ref}"] = t
pd.concat(inter).style.format({"slope_diff": "{:+.3f}", "p": "{:.2g}"})

### Paired comparison at each hop count

These are exact McNemar tests on the questions where exactly one arm of the pair is right.

In [ ]:
from scipy.stats import binomtest

wide = res.pivot(index="id", columns="model", values="correct").astype(bool).join(test_df.set_index("id").hops)
PAIRS = [("Laya (zero-shot)", "Phi-4-mini (direct)"), ("Laya (fine-tuned)", "Phi-4-mini (direct)"),
         ("Laya (fine-tuned)", "Phi-4-mini (CoT)"), ("Laya (zero-shot)", "Laya (fine-tuned)"),
         ("Phi-4-mini (direct)", "Phi-4-mini (CoT)")]
rows = []
for a, b in PAIRS:
    for h, g in wide.groupby("hops"):
        only_a, only_b = int((g[a] & ~g[b]).sum()), int((~g[a] & g[b]).sum())
        rows.append({"pair": f"{a} vs {b}", "hops": h, "diff_pts": 100 * (g[a].mean() - g[b].mean()),
                     "p": binomtest(only_a, only_a + only_b, 0.5).pvalue if only_a + only_b else 1.0})
mcnemar = pd.DataFrame(rows)
display(mcnemar.pivot(index="hops", columns="pair", values="diff_pts").round(1))
mcnemar.pivot(index="hops", columns="pair", values="p").style.format("{:.2g}").map(
    lambda p: "font-weight: bold" if p < 0.05 else "")

## 8. Figures

In [ ]:
COLORS = {"Laya (zero-shot)": "#9ecae9", "Laya (fine-tuned)": "#4C78A8",
          "Phi-4-mini (direct)": "#F58518", "Phi-4-mini (CoT)": "#B6451A"}
MARKERS = {"Laya (zero-shot)": "o", "Laya (fine-tuned)": "s", "Phi-4-mini (direct)": "^", "Phi-4-mini (CoT)": "D"}

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, (col, title) in zip(axes, [("accuracy", "threshold 0.5 (CoT: parsed answer)"),
                                   ("accuracy_tuned", "threshold tuned on the calibration slice")]):
    for m in MODELS:
        g = by_hops[by_hops.model == m].sort_values("hops")
        ax.plot(g.hops, g[col], marker=MARKERS[m], color=COLORS[m], lw=2, label=m)
        if col == "accuracy":
            ax.fill_between(g.hops, g.ci_low, g.ci_high, color=COLORS[m], alpha=0.15)
    ax.axhline(0.5, color="grey", ls="--", label="chance / 'not' shortcut (50%)")
    ax.set_xticks(HOPS); ax.set_ylim(0.3, 1.02); ax.grid(alpha=0.3)
    ax.set_xlabel("Reasoning hops (generator setting)")
    ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
    ax.set_title(f"Accuracy, {title}")
axes[0].set_ylabel("Accuracy"); axes[0].legend(loc="lower left", fontsize=9)
fig.suptitle(f"ProntoQA: accuracy vs reasoning hops ({len(test_df) // len(HOPS)} questions per hop count, "
             "balanced on label and negation; 95% Wilson bands)")
fig.savefig(FIG_DIR / "accuracy_vs_hops.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

ax = axes[0]   # threshold-free signal
for m in [m for m in MODELS if m != "Phi-4-mini (CoT)"]:
    g = by_hops[by_hops.model == m].sort_values("hops")
    ax.plot(g.hops, g.roc_auc, marker=MARKERS[m], color=COLORS[m], lw=2, label=m)
ax.axhline(0.5, color="grey", ls="--")
ax.set_xticks(HOPS); ax.set_ylim(0.3, 1.02); ax.set_xlabel("Hops"); ax.set_ylabel("ROC-AUC"); ax.legend(fontsize=8)
ax.set_title("ROC-AUC vs hops (no threshold)")

ax = axes[1]   # does an arm lean towards one answer as hops grow?
for m in MODELS:
    g = res[res.model == m].groupby("hops").pred.apply(lambda s: (s == "True").mean())
    ax.plot(g.index, g.values, marker=MARKERS[m], color=COLORS[m], lw=2, label=m)
ax.axhline(0.5, color="grey", ls="--", label="gold rate")
ax.set_xticks(HOPS); ax.set_ylim(0, 1); ax.set_xlabel("Hops"); ax.set_ylabel("Share answered True")
ax.set_title("Answer bias vs hops (threshold 0.5)")

ax = axes[2]   # per-label accuracy for each arm
for m in MODELS:
    g = by_hops[by_hops.model == m].sort_values("hops")
    ax.plot(g.hops, g.acc_true, marker=MARKERS[m], color=COLORS[m], lw=1.5, label=f"{m}: gold True")
    ax.plot(g.hops, g.acc_false, marker=MARKERS[m], color=COLORS[m], lw=1.5, ls=":", label=f"{m}: gold False")
ax.set_xticks(HOPS); ax.set_ylim(0, 1.02); ax.set_xlabel("Hops"); ax.set_ylabel("Accuracy"); ax.legend(fontsize=6)
ax.set_title("Accuracy by gold label (solid True, dotted False)")

plt.tight_layout()
fig.savefig(FIG_DIR / "signal_and_bias.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

ax = axes[0]
lat = res.groupby(["model", "hops"]).latency_s.mean().mul(1000).unstack(0)[list(MODELS)]
for m in MODELS:
    ax.plot(lat.index, lat[m], marker=MARKERS[m], color=COLORS[m], label=m)
ax.set_yscale("log"); ax.set_xticks(HOPS)
ax.set_xlabel("Hops"); ax.set_ylabel("ms per question (log scale)"); ax.legend(fontsize=8)
ax.set_title("Latency vs hops (Laya batch 1, Phi batch 8)")

ax = axes[1]
ax.boxplot([cot[cot.hops == h].output_tokens for h in HOPS], positions=HOPS, widths=0.6)
ax.axhline(COT_MAX_NEW_TOKENS, color="crimson", ls="--", label="token limit")
ax.set_xlabel("Hops"); ax.set_ylabel("Output tokens"); ax.legend()
ax.set_title("Phi-4-mini CoT reasoning length vs hops")

ax = axes[2]
h = pd.DataFrame(history)
if len(h):
    ax.plot(h.epoch, h.calib_acc, "o-", color=COLORS["Laya (fine-tuned)"], label="Laya held-out (calib) accuracy")
    for m, ls in [("Laya (zero-shot)", ":"), ("Phi-4-mini (CoT)", "--")]:
        ax.axhline(summary.loc[m, "accuracy"], color=COLORS[m], ls=ls, label=f"{m} (test)")
    ax.set_xticks(h.epoch); ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy"); ax.legend(fontsize=8)
ax.set_title(f"Laya fine-tuning ({train_meta['train_minutes']:.0f} min on this GPU)")

plt.tight_layout()
fig.savefig(FIG_DIR / "cost_and_finetuning.png", dpi=150, bbox_inches="tight"); plt.show()

### Deep questions where the arms disagree

These are 4- and 5-hop questions that fine-tuned Laya got wrong and Phi-4-mini CoT got right, shown with the gold proof and the CoT trace.

In [ ]:
by = res.set_index(["id", "model"])
deep = wide[(wide.hops >= 4) & ~wide["Laya (fine-tuned)"] & wide["Phi-4-mini (CoT)"]].index[:4]
tq = test_df.set_index("id")
for id_ in deep:
    r = tq.loc[id_]
    print("=" * 100)
    print(f"[{id_}] {r.hops} hops  GOLD: {r.label}\n{r.theory}\nStatement: {r.statement}\nGold proof: {r.gold_cot}\n")
    for m in MODELS:
        x = by.loc[(id_, m)]
        extra = f"  P(true)={x.p_true:.3f}" if pd.notna(x.get("p_true")) else ""
        print(f"  {'OK ' if x.correct else 'BAD'} {m:20s} -> {x.pred}{extra}")
    print("\n  CoT trace:\n  " + str(by.loc[(id_, "Phi-4-mini (CoT)")].output).replace("\n", "\n  ")[:1500])

## 9. Save results

Download the zip and commit `figures/`, `results.csv`, `by_hops.csv` and `summary.csv` to this experiment folder. The fine-tuned checkpoint is left out.

In [ ]:
import shutil
bundle = Path("/tmp/laya_vs_phi4_prontoqa_results")
if bundle.exists():
    shutil.rmtree(bundle)
shutil.copytree(CACHE_DIR, bundle / "cache")
shutil.copytree(FIG_DIR, bundle / "figures")
for name in ["results.csv", "by_hops.csv", "summary.csv"]:
    shutil.copy(BASE_DIR / name, bundle / name)
shutil.make_archive("laya_vs_phi4_prontoqa_results", "zip", bundle)
try:
    from google.colab import files
    files.download("laya_vs_phi4_prontoqa_results.zip")
except ImportError:
    print("Not on Colab; results in", BASE_DIR.resolve())